In [1]:

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

data_path = "../data/ebm_nlp_2_00"
DATA_DIR  = Path(data_path)

# ---- helper functions ----

def get_doc_ids(split="train", label_type="participants"):
    if split == "test":
        split = "test/gold"
    split_dir = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split
    return sorted([p.name.replace(".AGGREGATED.ann", "") for p in split_dir.glob("*.AGGREGATED.ann")])

def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    if split == "test":
        split = "test/gold"
    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split / f"{doc_id}.AGGREGATED.ann"
    if not ann_path.exists():
        return None
    with open(ann_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def count_all_zero_docs(doc_ids, label_type="participants", split="train"):
    all_zero = []
    for doc_id in doc_ids:
        labels = load_labels_for_doc(doc_id, label_type, split)
        if labels is None:
            continue
        if all(int(x) == 0 for x in labels):
            all_zero.append(doc_id)
    return all_zero

def assign_sentence_spans(tokens):
    spans = []
    start = 0
    n = len(tokens)
    for idx, token in enumerate(tokens):
        if token in {"?", "!"}:
            spans.append((start, idx + 1))
            start = idx + 1
        elif token == ".":
            # protect decimals like 65 . 5
            if idx > 0 and idx < n - 1 and tokens[idx-1].isdigit() and tokens[idx+1].isdigit():
                continue
            spans.append((start, idx + 1))
            start = idx + 1
    if start < n:
        spans.append((start, n))
    return spans

def assign_sentence_label(p_count, i_count, o_count):
    active = sum([p_count > 0, i_count > 0, o_count > 0])
    if active == 0:
        return "NONE"
    elif active > 1:
        return "MIXED"
    elif p_count > 0:
        return "P(articipant)"
    elif i_count > 0:
        return "I(ntervention)"
    else:
        return "O(utcome)"

# ---- build common doc ids ----

print("Loading document IDs...")
doc_ids_p = get_doc_ids("train", "participants")
doc_ids_i = get_doc_ids("train", "interventions")
doc_ids_o = get_doc_ids("train", "outcomes")

print("Filtering all-zero documents...")
all_zero_p = count_all_zero_docs(doc_ids_p, "participants", "train")
all_zero_i = count_all_zero_docs(doc_ids_i, "interventions", "train")
all_zero_o = count_all_zero_docs(doc_ids_o, "outcomes", "train")

modified_p = [d for d in doc_ids_p if d not in set(all_zero_p)]
modified_i = [d for d in doc_ids_i if d not in set(all_zero_i)]
modified_o = [d for d in doc_ids_o if d not in set(all_zero_o)]

common_doc_ids = sorted(set(modified_p) & set(modified_i) & set(modified_o))
print(f"✅ Common documents: {len(common_doc_ids)}")

# ---- build sentence records ----

print("Building sentence-level dataset (this takes a few minutes)...")

rows = []
for doc_id in common_doc_ids:
    tokens     = load_document(doc_id)
    p_labels   = load_labels_for_doc(doc_id, "participants",  "train")
    i_labels   = load_labels_for_doc(doc_id, "interventions", "train")
    o_labels   = load_labels_for_doc(doc_id, "outcomes",      "train")

    if p_labels is None or i_labels is None or o_labels is None:
        continue

    spans = assign_sentence_spans(tokens)

    for sent_id, (start, end) in enumerate(spans):
        sent_tokens = tokens[start:end]
        sent_p      = p_labels[start:end]
        sent_i      = i_labels[start:end]
        sent_o      = o_labels[start:end]

        p_count = sum(1 for x in sent_p if int(x) != 0)
        i_count = sum(1 for x in sent_i if int(x) != 0)
        o_count = sum(1 for x in sent_o if int(x) != 0)

        gold_label   = assign_sentence_label(p_count, i_count, o_count)
        sentence_text = " ".join(sent_tokens)

        rows.append({
            "doc_id":        doc_id,
            "sentence_id":   sent_id,
            "sentence_text": sentence_text,
            "gold_label":    gold_label,
            "p_count":       p_count,
            "i_count":       i_count,
            "o_count":       o_count
        })

sentence_records_df = pd.DataFrame(rows)
print(f"✅ Built {len(sentence_records_df)} sentences from {len(common_doc_ids)} documents")
print(f"\nLabel distribution:")
print(sentence_records_df["gold_label"].value_counts())

# ---- save ----
sentence_records_df.to_csv("../outputs/sentence_records_df.csv", index=False)
print("\n✅ Saved to outputs/sentence_records_df.csv")

Loading document IDs...
Filtering all-zero documents...
✅ Common documents: 3895
Building sentence-level dataset (this takes a few minutes)...
✅ Built 42481 sentences from 3895 documents

Label distribution:
gold_label
MIXED             12662
O(utcome)         10669
NONE              10360
I(ntervention)     5966
P(articipant)      2824
Name: count, dtype: int64

✅ Saved to outputs/sentence_records_df.csv
